<a href="https://colab.research.google.com/github/Aaryant74/Data_Science_Assignments/blob/main/20_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1: Identify the Business Problem**

From the file name monthly_milk_production.csv, it is clear that this dataset showing the monthly milk production of a dairy farm or industry.

**Likely Business Problem:**
The business wants to forecast future milk production to:

•	Ensure proper supply chain and distribution planning.

•	Manage inventory and storage efficiently.

•	Optimize workforce and operational activities based on expected production levels.

•	Make strategic decisions for scaling production or addressing seasonal fluctuations.


**Step 2: Define the Objective**

**Objective Statement:**

The objective is to develop a time series forecasting model that accurately predicts the monthly milk production for the upcoming months. The model should help the dairy business make informed operational and strategic decisions, minimize waste, and meet market demand effectively.


**1.	Exploratory Data Analysis (EDA)**

* Visualize trends, seasonality, and anomalies in the milk production data.
* Check for any missing values or outliers.
* Normalize or scale the data for neural network models.

In [ ]:
# Loading dataset
import pandas as pd

# Load CSV file
df = pd.read_csv('/content/monthly_milk_production.csv')

# Show first rows
df.head()

In [ ]:
# Check Data Info & Missing Values
# Basic info (data types, null values)
df.info()

# Check missing values
print(df.isnull().sum())

In [ ]:
# Convert Date Column (IMPORTANT for time series)
# Convert Month column to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Set it as index (important for plotting)
df.set_index('Date', inplace=True)

df.head()

In [ ]:
# Plot Milk Production (Trend + Seasonality)
import matplotlib.pyplot as plt

# Line plot to see trend and seasonality
plt.figure(figsize=(10,5))
plt.plot(df, label='Milk Production')
plt.title("Milk Production Over Time")
plt.xlabel("Date")
plt.ylabel("Production")
plt.legend()
plt.show()

# Shows:

# Trend (overall increase/decrease)
# Seasonality (repeating patterns)

In [ ]:
# Check Seasonality (Monthly Pattern)
# Extract month from index
df['Month_num'] = df.index.month

# Average production per month
monthly_avg = df.groupby('Month_num').mean()

monthly_avg.plot(kind='bar')
plt.title("Average Milk Production per Month")
plt.xlabel("Month")
plt.ylabel("Production")
plt.show()

# Helps identify seasonal behavior

In [ ]:
# Detect Outliers
# Boxplot to detect outliers
df.boxplot()
plt.title("Boxplot for Outlier Detection")
plt.show()

In [ ]:
# Normalize/Scale Data
# Needed for neural networks
from sklearn.preprocessing import MinMaxScaler

# Initialize scaler (values between 0 and 1)
scaler = MinMaxScaler()

# Scale the production values
df['Scaled_Production'] = scaler.fit_transform(df[['Production']])

df.head()

In [ ]:
# Plot Scaled Data
plt.figure(figsize=(10,5))
plt.plot(df['Scaled_Production'], label='Scaled Production')
plt.title("Normalized Milk Production")
plt.legend()
plt.show()

**2.	Data Preparation for Deep Learning**

* Create input-output sequences (time windows) suitable for training RNNs/LSTMs/GRUs.
* Split data into training, validation, and test sets.
* Reshape data for model input dimensions.

In [ ]:
# Creating Time Sequences
import numpy as np

# Function to create sequences
def create_sequences(data, seq_length):
    X = []
    y = []

    for i in range(len(data) - seq_length):
        # Input sequence (past values)
        X.append(data[i:i+seq_length])

        # Output (next value)
        y.append(data[i+seq_length])

    return np.array(X), np.array(y)

# Use scaled data
data = df['Scaled_Production'].values

# Define window size (e.g., 12 months)
seq_length = 12

# Create sequences
X, y = create_sequences(data, seq_length)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Split into Train, Validation, Test
# Total number of samples
total_size = len(X)

# 70% train, 15% validation, 15% test
train_size = int(total_size * 0.7)
val_size = int(total_size * 0.15)

# Split data
X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size+val_size]
y_val = y[train_size:train_size+val_size]

X_test = X[train_size+val_size:]
y_test = y[train_size+val_size:]

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

In [ ]:
# Reshape Data for LSTM
# Reshape into 3D format
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print("Reshaped Train:", X_train.shape)
# 1 = number of features (only milk production)

**3.	Model Building**

* Build three separate models:

 a. Basic RNN

 b. LSTM

 c. GRU

* Tune hyperparameters (e.g., window size, number of units, batch size, epochs).

* Use appropriate loss functions and optimizers.

In [ ]:
# Importing libraries
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, GRU

In [ ]:
# Creating Basic RNN model
rnn_model = Sequential()

# SimpleRNN layer with 50 units (neurons)
rnn_model.add(SimpleRNN(50, activation='tanh', input_shape=(X_train.shape[1], 1)))

# Output layer (1 value prediction)
rnn_model.add(Dense(1))

# Compile model
rnn_model.compile(
    optimizer='adam',       # optimizer for updating weights
    loss='mse'              # mean squared error (best for regression)
)

# Train model
rnn_history = rnn_model.fit(
    X_train, y_train,
    epochs=20,              # number of training iterations
    batch_size=16,          # samples per batch
    validation_data=(X_val, y_val)
)

In [ ]:
# Creating LSTM model
lstm_model = Sequential()

# LSTM layer (better for long-term dependencies)
lstm_model.add(LSTM(50, activation='tanh', input_shape=(X_train.shape[1], 1)))

# Output layer
lstm_model.add(Dense(1))

# Compile model
lstm_model.compile(
    optimizer='adam',
    loss='mse'
)

# Train model
lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_val, y_val)
)

In [ ]:
# Creating GRU model
gru_model = Sequential()

# GRU layer (faster than LSTM, similar performance)
gru_model.add(GRU(50, activation='tanh', input_shape=(X_train.shape[1], 1)))

# Output layer
gru_model.add(Dense(1))

# Compile model
gru_model.compile(
    optimizer='adam',
    loss='mse'
)

# Train model
gru_history = gru_model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_val, y_val)
)

In [ ]:
# Hyperparameter Tuning
# Example tuned LSTM model
lstm_tuned = Sequential()

# Increased units (more learning capacity)
lstm_tuned.add(LSTM(100, activation='tanh', input_shape=(X_train.shape[1], 1)))

lstm_tuned.add(Dense(1))

lstm_tuned.compile(
    optimizer='adam',
    loss='mse'
)

# Increased epochs and batch size
lstm_tuned.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val)
)

**4.	Model Evaluation**

* Plot predictions vs. actual values.
* Calculate forecasting metrics: RMSE, MAE, MAPE.
* Compare the performance of RNN, LSTM, and GRU.


In [ ]:
# Predictions
# Predict values using all models

rnn_pred = rnn_model.predict(X_test)
lstm_pred = lstm_model.predict(X_test)
gru_pred = gru_model.predict(X_test)

In [ ]:
# Plot Predictions vs Actual Values
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

# Actual values
plt.plot(y_test, label='Actual')

# Model predictions
plt.plot(rnn_pred, label='RNN')
plt.plot(lstm_pred, label='LSTM')
plt.plot(gru_pred, label='GRU')

plt.title("Model Comparison: Actual vs Predictions")
plt.xlabel("Time")
plt.ylabel("Milk Production")
plt.legend()
plt.show()

In [ ]:
# Evaluation Metrics (RMSE, MAE, MAPE)
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# RMSE
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# MAPE
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

# Calculate metrics for each model
rnn_rmse = rmse(y_test, rnn_pred)
lstm_rmse = rmse(y_test, lstm_pred)
gru_rmse = rmse(y_test, gru_pred)

rnn_mae = mean_absolute_error(y_test, rnn_pred)
lstm_mae = mean_absolute_error(y_test, lstm_pred)
gru_mae = mean_absolute_error(y_test, gru_pred)

rnn_mape = mape(y_test, rnn_pred)
lstm_mape = mape(y_test, lstm_pred)
gru_mape = mape(y_test, gru_pred)

# Print results
print("RNN -> RMSE:", rnn_rmse, "MAE:", rnn_mae, "MAPE:", rnn_mape)
print("LSTM -> RMSE:", lstm_rmse, "MAE:", lstm_mae, "MAPE:", lstm_mape)
print("GRU -> RMSE:", gru_rmse, "MAE:", gru_mae, "MAPE:", gru_mape)

In [ ]:
# Compare Results
results = pd.DataFrame({
    'Model': ['RNN', 'LSTM', 'GRU'],
    'RMSE': [rnn_rmse, lstm_rmse, gru_rmse],
    'MAE': [rnn_mae, lstm_mae, gru_mae],
    'MAPE': [rnn_mape, lstm_mape, gru_mape]
})

print(results)

**5.	Prediction and Visualization**

* Forecast milk production for the next 12 months.
* Visualize the predicted trend with uncertainty or confidence intervals if possible.

In [ ]:
# Forecast Next 12 Months
# Take last sequence from dataset
last_sequence = X_test[-1]

# Store predictions
future_predictions = []

# Predict next 12 months
for _ in range(12):
    # Reshape for model input
    last_sequence_reshaped = last_sequence.reshape((1, last_sequence.shape[0], 1))

    # Predict next value (using LSTM - best model)
    next_pred = lstm_model.predict(last_sequence_reshaped)

    # Store prediction
    future_predictions.append(next_pred[0][0])

    # Update sequence (remove first value, add new prediction)
    last_sequence = np.append(last_sequence[1:], next_pred)

# Convert to array
future_predictions = np.array(future_predictions)

print(future_predictions)

In [ ]:
# Convert Back to Original Scale
# Reshape for inverse transform
future_predictions = future_predictions.reshape(-1, 1)

# Convert scaled values back to original milk production
future_predictions_actual = scaler.inverse_transform(future_predictions)

print(future_predictions_actual)

In [ ]:
# Create Future Dates
# Get last date from dataset
last_date = df.index[-1]

# Generate next 12 months
future_dates = pd.date_range(start=last_date, periods=13, freq='M')[1:]

In [ ]:
# Plot Forecast
plt.figure(figsize=(12,6))

# Plot actual data
plt.plot(df.index, df['Production'], label='Actual Data')

# Plot future predictions
plt.plot(future_dates, future_predictions_actual, label='Forecast (Next 12 Months)', color='red')

plt.title("Milk Production Forecast")
plt.xlabel("Date")
plt.ylabel("Production")
plt.legend()
plt.show()

In [ ]:
# Add Confidence Interval (Simple Approximation)
# Neural networks don’t directly give confidence intervals, so we approximate using error margin (based on RMSE)
# Use LSTM RMSE as uncertainty
upper_bound = future_predictions_actual.flatten() + lstm_rmse
lower_bound = future_predictions_actual.flatten() - lstm_rmse

plt.figure(figsize=(12,6))

# Actual data
plt.plot(df.index, df['Production'], label='Actual')

# Forecast
plt.plot(future_dates, future_predictions_actual, label='Forecast', color='red')

# Confidence interval
plt.fill_between(future_dates, lower_bound, upper_bound, color='pink', alpha=0.3)

plt.title("Forecast with Confidence Interval")
plt.legend()
plt.show()

**6.	Business Insights**

Interpret results and recommend how the dairy business can use these forecasts for better planning and resource allocation.


**Interpretation of Results :**

A. The forecasting models (especially LSTM/GRU) successfully captured trend and seasonal patterns in milk production.

B. The predictions indicate that milk production follows a cyclical pattern, with certain months showing higher or lower output.

C. The relatively low error metrics (RMSE, MAE, MAPE) suggest that the model provides reliable forecasts for future planning.

D. The confidence interval shows that while predictions are accurate, there is some uncertainty, which is normal in real-world forecasting.

**Business Recoomendations :**

I. Production Planning

II. Inventory Management

III. Supply Chain Management

IV. Resource Allocation

V. Pricing Startegy

VI. Risk Management

**Conclusion :**
The forecasting model enables data-driven decision-making, helping dairy businesses optimize production, reduce waste, and improve profitability through better planning and resource allocation.